**How to Finetune LLMs with LoRA:**

In [6]:
%pip install -U peft transformers datasets accelerate
%pip uninstall -y torchao
!mkdir -p ../cache/working

In [7]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
import transformers
from transformers import TrainingArguments, Trainer
import os
import time
import torch

In [8]:
model_name = "bigscience/bloomz-560m"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

foundation_model = AutoModelForCausalLM.from_pretrained(model_name)
foundation_model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [9]:
data = load_dataset("Abirate/english_quotes", split="train[:10%]")

def tokenize(example):
    tokens = tokenizer(
        example["quote"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

data = data.map(
    tokenize,
    remove_columns=data.column_names
)

train_sample = data.select(range(5))
display(train_sample)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5
})

In [10]:
lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176


In [11]:
print(len(data))
print(data[0])

251
{'input_ids': [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1502, 17143, 33218, 30, 39839, 4384, 632, 11226, 15713, 17, 982], 'attention_mask': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1502, 17143, 33218, 30, 39839, 4384, 632, 11226, 15713, 17, 982]}


In [12]:
import os
import transformers
from transformers import TrainingArguments, Trainer

print("Dataset size:", len(data))
print("Sample:", data[0])

print("\nTrainable params:")
peft_model.print_trainable_parameters()

output_directory = os.path.join("../cache/working", "peft_lab_outputs")

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    per_device_train_batch_size=1,
    learning_rate=3e-2,
    num_train_epochs=1,
    use_cpu=True,
    logging_steps=1,
    logging_strategy="steps",
    disable_tqdm=False,
    max_steps=5,
    remove_unused_columns=False
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.default_data_collator
)

print("\nStarting training...")

trainer.train()

print("\nTraining finished ✅")

Dataset size: 251
Sample: {'input_ids': [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1502, 17143, 33218, 30, 39839, 4384, 632, 11226, 15713, 17, 982], 'attention_mask': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1502, 17143, 33218, 30, 39839, 4384, 632, 11226, 15713, 17, 982]}

Trainable params:
trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176

Starting training...


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000



Training finished ✅


In [13]:
# output_directory = os.path.join("../cache/working", "peft_lab_outputs")

# training_args = TrainingArguments(
#     report_to="none",
#     output_dir=output_directory,
#     auto_find_batch_size=True,
#     learning_rate=3e-2,
#     num_train_epochs=1,
#     use_cpu=True,
#     logging_steps=10,
#     logging_strategy="steps"
# )

# trainer = Trainer(
#     model=peft_model,
#     args=training_args,
#     train_dataset=data,
#     data_collator=transformers.default_data_collator
#     # data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
# )

# trainer.train()

In [17]:
time_now = int(time.time())

peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

print("Saved LoRA model to:", peft_model_path)

Saved LoRA model to: ../cache/working/peft_lab_outputs/peft_model_1777914105


In [18]:
base_model_for_inference = AutoModelForCausalLM.from_pretrained(model_name)
base_model_for_inference.config.pad_token_id = tokenizer.pad_token_id

loaded_peft_model = PeftModel.from_pretrained(
    base_model_for_inference,
    peft_model_path,
    is_trainable=False
)

loaded_peft_model.eval()

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(250880, 1024)
        (word_embeddings_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0-23): 24 x BloomBlock(
            (input_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=3072, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=1, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=1, out_features=3072, bias=False)
                )
                (lora_e

In [21]:
inputs = tokenizer("Quote about life:\n", return_tensors="pt")

outputs = loaded_peft_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

['Quote about life:\n']
